# Matched QGECAF gate controls on the locked patient folds

This runner trains G0, G1 and G3 on the exact saved QGECAF fold partitions and reuses G2 from the completed experiment. It loads each fold's saved BGWO masks, scalers, PCA objects and quality scaler without refitting them. The untouched final-test cohort is excluded.

Keep this notebook, `QGECAF_Matched_Gate_Controls_Automated.py`, and `QGECAF_Reviewer_Revision_Nested_BGWO.py` in the same folder. Restart the kernel before running this experiment.

In [1]:
import os
os.environ['TF_DETERMINISTIC_OPS'] = '1'

from pathlib import Path
import pandas as pd
from QGECAF_Matched_Gate_Controls_Automated import Config, run
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.10.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
package_root = Path.cwd()
cfg = Config(
    qgecaf_root=(
        r'D:\chaitali\qgacef'
        r'\QGECAF_REVIEWER_REVISION_NESTED_BGWO'
    ),
    qgecaf_source=str(package_root / 'QGECAF_Reviewer_Revision_Nested_BGWO.py'),
    output=(
        r'D:\chaitali\qgacef'
        r'\QGECAF_MATCHED_GATE_CONTROLS'
    ),
    image_size=128,
    num_classes=6,
    outer_folds=10,
    seed=42,
    embed_dim=64,
    attention_heads=4,
    batch_size=8,
    epochs=50,
    learning_rate=1e-4,
    kl_weight=1e-3,
    dropout=0.30,
    fine_tune_last=30,
    early_stopping_patience=8,
    fixed_gate_lambda=0.5,
    resume=True,
)
cfg

Config(qgecaf_root='D:\\chaitali\\qgacef\\QGECAF_REVIEWER_REVISION_NESTED_BGWO', qgecaf_source='D:\\chaitali\\qgacef\\QGECAF_Reviewer_Controls_Automated_Package\\QGECAF_Reviewer_Revision_Nested_BGWO.py', output='D:\\chaitali\\qgacef\\QGECAF_MATCHED_GATE_CONTROLS', image_size=128, num_classes=6, outer_folds=10, seed=42, embed_dim=64, attention_heads=4, batch_size=8, epochs=50, learning_rate=0.0001, kl_weight=0.001, dropout=0.3, fine_tune_last=30, early_stopping_patience=8, gate_hidden_units=16, fixed_gate_lambda=0.5, ece_bins=15, lower_quality_percentile=33.0, upper_quality_percentile=67.0, resume=True, smoke_test=False)

In [3]:
root = Path(cfg.qgecaf_root)
required = [
    Path(cfg.qgecaf_source),
    root / 'locked_patient_fold_assignments.csv',
    root / 'raw_features_presplit.npz',
    root / 'fold_metrics.csv',
    root / 'class_names.json',
]
for fold in range(1, 11):
    fold_root = root / 'cross_validation' / f'fold_{fold:02d}'
    required.extend([
        fold_root / 'patient_partition_audit.csv',
        fold_root / 'preprocessing.joblib',
        fold_root / 'outer_holdout_predictions.csv',
    ])
missing = [path for path in required if not path.is_file()]
print('Required files:', len(required))
print('Missing files:', len(missing))
for path in missing[:20]:
    print('MISSING:', path)
assert not missing, 'Resolve the missing files before training gate controls.'
print('PASS: Saved preprocessing and all ten matched patient folds are available.')

Required files: 35
Missing files: 0
PASS: Saved preprocessing and all ten matched patient folds are available.


## Run or safely resume

This performs 30 new training runs: G0, G1 and G3 across ten folds. G2 is read from the completed QGECAF results and is not retrained. Completed variant-folds are verified and skipped.

In [4]:
run(cfg)

TensorFlow 2.10.0; GPUs detected: 1

MATCHED GATE CONTROLS — OUTER FOLD 1/10

--- G0_fixed_0p5 ---
Completed fold verified; skipping retraining.

--- G1_learned_global ---
Completed fold verified; skipping retraining.

--- G3_shuffled_quality ---
Completed fold verified; skipping retraining.

MATCHED GATE CONTROLS — OUTER FOLD 2/10

--- G0_fixed_0p5 ---
Completed fold verified; skipping retraining.

--- G1_learned_global ---
Completed fold verified; skipping retraining.

--- G3_shuffled_quality ---
Completed fold verified; skipping retraining.

MATCHED GATE CONTROLS — OUTER FOLD 3/10

--- G0_fixed_0p5 ---
Completed fold verified; skipping retraining.

--- G1_learned_global ---
Completed fold verified; skipping retraining.

--- G3_shuffled_quality ---
Completed fold verified; skipping retraining.

MATCHED GATE CONTROLS — OUTER FOLD 4/10

--- G0_fixed_0p5 ---
Completed fold verified; skipping retraining.

--- G1_learned_global ---
Completed fold verified; skipping retraining.

--- G3_shu

In [5]:
result_root = Path(cfg.output)
display(pd.read_csv(result_root / 'gate_control_mean_sd_ci95.csv'))
display(pd.read_csv(result_root / 'G2_vs_controls_wilcoxon_holm.csv'))
display(pd.read_csv(result_root / 'quality_stratified_gate_controls.csv').head(24))
print('Run complete:', (result_root / 'RUN_COMPLETE.json').is_file())

,variant,metric,folds,mean,standard_deviation,ci95_low,ci95_high
0,G0_fixed_0p5,accuracy,10,0.929242,0.012791,0.920092,0.938393
1,G0_fixed_0p5,balanced_accuracy,10,0.932634,0.012438,0.923736,0.941532
2,G0_fixed_0p5,macro_f1,10,0.915906,0.012673,0.906840,0.924971
3,G0_fixed_0p5,weighted_f1,10,0.929527,0.012781,0.920384,0.938670
4,G0_fixed_0p5,mcc,10,0.910134,0.014741,0.899589,0.920680
5,G0_fixed_0p5,ece_15_bins,10,0.028741,0.017065,0.016534,0.040949
6,G0_fixed_0p5,brier_multiclass,10,0.117339,0.018828,0.103870,0.130808
7,G0_fixed_0p5,negative_log_likelihood,10,0.293419,0.040758,0.264262,0.322575
8,G0_fixed_0p5,uncertainty_error_auroc,10,0.736227,0.064367,0.690182,0.782273
9,G1_learned_global,accuracy,10,0.929736,0.010952,0.921901,0.937571


,metric,reference,comparison,folds,mean_difference_G2_minus_control,wilcoxon_statistic,raw_p,rank_biserial_G2_minus_control,holm_adjusted_p_within_metric,significant_at_0.05
0,accuracy,G2_correct_quality,G0_fixed_0p5,10,-0.003267,21.0,0.556641,-0.236364,1.000000,False
1,accuracy,G2_correct_quality,G1_learned_global,10,-0.003760,13.0,0.160156,-0.527273,0.480469,False
2,accuracy,G2_correct_quality,G3_shuffled_quality,10,-0.002509,23.0,0.695312,-0.163636,1.000000,False
3,balanced_accuracy,G2_correct_quality,G0_fixed_0p5,10,0.003329,19.0,0.431641,0.309091,0.863281,False
4,balanced_accuracy,G2_correct_quality,G1_learned_global,10,-0.002887,16.0,0.275391,-0.418182,0.826172,False
5,balanced_accuracy,G2_correct_quality,G3_shuffled_quality,10,-0.000730,25.0,0.845703,0.090909,0.863281,False
6,macro_f1,G2_correct_quality,G0_fixed_0p5,10,-0.004010,20.0,0.492188,-0.272727,0.984375,False
7,macro_f1,G2_correct_quality,G1_learned_global,10,-0.006272,10.0,0.083984,-0.636364,0.251953,False
8,macro_f1,G2_correct_quality,G3_shuffled_quality,10,-0.004444,21.0,0.556641,-0.236364,0.984375,False
9,weighted_f1,G2_correct_quality,G0_fixed_0p5,10,-0.002770,21.0,0.556641,-0.236364,1.000000,False


,variant,fold,quality_group,inner_train_q33,inner_train_q67,images,patients,classes_present,missing_classes,accuracy,macro_f1,negative_log_likelihood,brier_multiclass,ece_15_bins,gate_lambda_mean,gate_lambda_sample_SD,gate_lambda_median,gate_lambda_IQR
0,G0_fixed_0p5,1,High,0.443586,0.516497,264,81,6,NaN,0.965909,0.930203,0.196907,0.071399,0.072058,0.5,0.0,0.5,0.0
1,G0_fixed_0p5,1,Low,0.443586,0.516497,416,112,6,NaN,0.959135,0.951229,0.201640,0.070607,0.062400,0.5,0.0,0.5,0.0
2,G0_fixed_0p5,1,Medium,0.443586,0.516497,331,99,6,NaN,0.933535,0.924525,0.287343,0.113590,0.056510,0.5,0.0,0.5,0.0
3,G0_fixed_0p5,2,High,0.442399,0.519177,330,94,6,NaN,0.945455,0.867165,0.257061,0.093014,0.056847,0.5,0.0,0.5,0.0
4,G0_fixed_0p5,2,Low,0.442399,0.519177,381,109,6,NaN,0.897638,0.893773,0.365371,0.155336,0.043050,0.5,0.0,0.5,0.0
5,G0_fixed_0p5,2,Medium,0.442399,0.519177,412,103,6,NaN,0.939320,0.918789,0.239837,0.097860,0.059037,0.5,0.0,0.5,0.0
6,G0_fixed_0p5,3,High,0.442699,0.518401,455,92,6,NaN,0.938462,0.891206,0.240773,0.095522,0.061704,0.5,0.0,0.5,0.0
7,G0_fixed_0p5,3,Low,0.442699,0.518401,355,116,6,NaN,0.935211,0.932491,0.233336,0.091950,0.068569,0.5,0.0,0.5,0.0
8,G0_fixed_0p5,3,Medium,0.442699,0.518401,442,118,6,NaN,0.902715,0.885663,0.363526,0.153340,0.036415,0.5,0.0,0.5,0.0
9,G0_fixed_0p5,4,High,0.443100,0.517465,276,89,6,NaN,0.938406,0.923374,0.266548,0.104280,0.026802,0.5,0.0,0.5,0.0


Run complete: True


In [6]:
%pip install python-docx

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement python-docx (from versions: none)
ERROR: No matching distribution found for python-docx


In [7]:
from pathlib import Path

import pandas as pd
from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT, WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt


# =====================================================================
# 1. PATHS
# =====================================================================

RESULT_ROOT = Path(
    r"D:\chaitali\qgacef\QGECAF_MATCHED_GATE_CONTROLS"
)

OUTPUT_WORD_FILE = (
    RESULT_ROOT / "QGECAF_Matched_Gate_Controls_Results.docx"
)

print("Result folder exists:", RESULT_ROOT.is_dir())
print("Output Word file:", OUTPUT_WORD_FILE)


# =====================================================================
# 2. RESULT FILES
# =====================================================================

# Main tables will appear first.
MAIN_TABLES = {
    "Mean, Standard Deviation and 95% Confidence Intervals":
        "gate_control_mean_sd_ci95.csv",

    "Friedman Omnibus Tests":
        "gate_controls_friedman_tests.csv",

    "Paired G2 Versus Control Comparisons":
        "G2_vs_controls_wilcoxon_holm.csv",

    "Leakage-Free Quality Thresholds by Fold":
        "quality_thresholds_by_fold.csv",
}

# Detailed tables will appear as appendices.
APPENDIX_TABLES = {
    "Appendix A: Fold-Wise Gate-Control Metrics":
        "all_gate_control_fold_metrics.csv",

    "Appendix B: Quality-Stratified Gate-Control Results":
        "quality_stratified_gate_controls.csv",
}


# =====================================================================
# 3. FORMATTING FUNCTIONS
# =====================================================================

def format_value(value):
    """Format values consistently for Word tables."""

    if pd.isna(value):
        return "NA"

    if isinstance(value, bool):
        return "Yes" if value else "No"

    if isinstance(value, int):
        return str(value)

    if isinstance(value, float):
        if value != 0 and abs(value) < 0.0001:
            return f"{value:.3e}"
        return f"{value:.6f}"

    return str(value)


def shade_cell(cell, colour):
    """Apply a background colour to a Word table cell."""

    properties = cell._tc.get_or_add_tcPr()
    shading = OxmlElement("w:shd")
    shading.set(qn("w:fill"), colour)
    properties.append(shading)


def set_repeat_table_header(row):
    """Repeat the table header at the top of subsequent pages."""

    row_properties = row._tr.get_or_add_trPr()
    repeat_header = OxmlElement("w:tblHeader")
    repeat_header.set(qn("w:val"), "true")
    row_properties.append(repeat_header)


def prevent_row_splitting(row):
    """Prevent one table row from splitting across two pages."""

    row_properties = row._tr.get_or_add_trPr()
    cannot_split = OxmlElement("w:cantSplit")
    row_properties.append(cannot_split)


def set_cell_text(cell, text, bold=False, font_size=7):
    """Insert formatted text into a Word table cell."""

    cell.text = ""

    paragraph = cell.paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

    run = paragraph.add_run(str(text))
    run.bold = bold
    run.font.name = "Times New Roman"
    run.font.size = Pt(font_size)

    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER


def add_dataframe_table(
    document,
    dataframe,
    font_size=7,
    maximum_rows=None,
):
    """Insert a pandas DataFrame as a formatted Word table."""

    dataframe = dataframe.copy()

    if maximum_rows is not None and len(dataframe) > maximum_rows:
        dataframe = dataframe.head(maximum_rows)

    table = document.add_table(
        rows=1,
        cols=len(dataframe.columns),
    )

    table.style = "Table Grid"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = True

    # Header
    header_row = table.rows[0]
    set_repeat_table_header(header_row)

    for column_index, column_name in enumerate(dataframe.columns):
        readable_name = str(column_name).replace("_", " ")

        set_cell_text(
            header_row.cells[column_index],
            readable_name,
            bold=True,
            font_size=font_size,
        )

        shade_cell(
            header_row.cells[column_index],
            "D9EAF7",
        )

    # Body
    for _, dataframe_row in dataframe.iterrows():
        word_row = table.add_row()
        prevent_row_splitting(word_row)

        for column_index, value in enumerate(dataframe_row):
            set_cell_text(
                word_row.cells[column_index],
                format_value(value),
                bold=False,
                font_size=font_size,
            )

    document.add_paragraph()

    return table


def add_heading(document, heading_text, level=1):
    """Add a Times New Roman heading."""

    heading = document.add_heading(
        heading_text,
        level=level,
    )

    for run in heading.runs:
        run.font.name = "Times New Roman"

    return heading


def add_source_note(document, filename, dataframe):
    """Add the source filename and row count."""

    paragraph = document.add_paragraph()

    run = paragraph.add_run(
        f"Source: {filename}; rows: {len(dataframe)}."
    )

    run.italic = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(8)


# =====================================================================
# 4. CREATE THE DOCUMENT
# =====================================================================

document = Document()

section = document.sections[0]
section.orientation = WD_ORIENT.LANDSCAPE
section.page_width = Inches(11.69)
section.page_height = Inches(8.27)

section.top_margin = Inches(0.50)
section.bottom_margin = Inches(0.50)
section.left_margin = Inches(0.40)
section.right_margin = Inches(0.40)

normal_style = document.styles["Normal"]
normal_style.font.name = "Times New Roman"
normal_style.font.size = Pt(9)


# =====================================================================
# 5. TITLE
# =====================================================================

title = document.add_heading(
    "QGECAF Matched Gate-Control Results",
    level=0,
)

title.alignment = WD_ALIGN_PARAGRAPH.CENTER

for run in title.runs:
    run.font.name = "Times New Roman"
    run.font.size = Pt(16)
    run.bold = True

subtitle = document.add_paragraph()
subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER

subtitle_run = subtitle.add_run(
    "Patient-Level Ten-Fold Cross-Validation"
)

subtitle_run.bold = True
subtitle_run.font.name = "Times New Roman"
subtitle_run.font.size = Pt(10)


# =====================================================================
# 6. EXPERIMENT DESCRIPTION
# =====================================================================

add_heading(document, "Experimental Protocol", level=1)

document.add_paragraph(
    "The matched gate-control experiment compared four fusion strategies "
    "under the same locked patient-level outer folds. G0 used a fixed fusion "
    "weight of 0.5, G1 used one learned global fusion weight, G2 used the "
    "correct sample-specific quality gate, and G3 used the same "
    "sample-specific gate architecture with shuffled quality inputs. "
    "The saved fold-specific BGWO masks, StandardScaler objects, PCA "
    "transformations, quality scaler, model architecture, evidential loss, "
    "training settings and outer-holdout patients were held fixed. The "
    "untouched final-test cohort was not evaluated in this control experiment."
)

control_table = pd.DataFrame({
    "Control": [
        "G0_fixed_0p5",
        "G1_learned_global",
        "G2_correct_quality",
        "G3_shuffled_quality",
    ],
    "Gate": [
        "Fixed lambda = 0.5",
        "One learned global lambda",
        "Sample-specific gate using correct quality",
        "Sample-specific gate using shuffled quality",
    ],
    "Purpose": [
        "Tests equal averaging",
        "Tests one dataset-level weighting",
        "Tests the proposed quality-conditioned gate",
        "Tests whether correct quality correspondence is informative",
    ],
})

add_dataframe_table(
    document,
    control_table,
    font_size=8,
)


# =====================================================================
# 7. MAIN RESULT TABLES
# =====================================================================

included_files = []
missing_files = []

for section_heading, filename in MAIN_TABLES.items():

    csv_path = RESULT_ROOT / filename

    if not csv_path.is_file():
        missing_files.append(filename)
        continue

    dataframe = pd.read_csv(csv_path)

    add_heading(
        document,
        section_heading,
        level=1,
    )

    add_source_note(
        document,
        filename,
        dataframe,
    )

    font_size = 6 if len(dataframe.columns) > 8 else 7

    add_dataframe_table(
        document,
        dataframe,
        font_size=font_size,
    )

    included_files.append(filename)


# =====================================================================
# 8. INTERPRETATION GUIDANCE
# =====================================================================

add_heading(document, "Interpretation Guidance", level=1)

interpretation_notes = [
    (
        "For accuracy, balanced accuracy, macro F1, weighted F1, MCC and "
        "uncertainty–error AUROC, higher values are preferred."
    ),
    (
        "For expected calibration error, multiclass Brier score and negative "
        "log-likelihood, lower values are preferred."
    ),
    (
        "The column mean_difference_G2_minus_control represents the paired "
        "fold-wise mean difference. Its direction must be interpreted "
        "according to whether higher or lower values are preferred."
    ),
    (
        "A gate-control difference should be described as statistically "
        "significant only when the Holm-adjusted p-value is below 0.05."
    ),
    (
        "A non-significant result does not demonstrate equivalence or "
        "non-inferiority between the configurations."
    ),
    (
        "The comparison of G2 with G3 is the primary test of whether the "
        "correct image-to-quality relationship provides measurable value."
    ),
]

for note in interpretation_notes:
    paragraph = document.add_paragraph(style="List Bullet")
    paragraph.add_run(note)


# =====================================================================
# 9. DETAILED APPENDICES
# =====================================================================

for section_heading, filename in APPENDIX_TABLES.items():

    csv_path = RESULT_ROOT / filename

    if not csv_path.is_file():
        missing_files.append(filename)
        continue

    dataframe = pd.read_csv(csv_path)

    document.add_page_break()

    add_heading(
        document,
        section_heading,
        level=1,
    )

    add_source_note(
        document,
        filename,
        dataframe,
    )

    if filename == "quality_stratified_gate_controls.csv":
        document.add_paragraph(
            "The quality thresholds were calculated from the corresponding "
            "inner-training partition and then applied unchanged to the "
            "outer-holdout images. The outer holdout did not determine its "
            "own quality thresholds."
        )

    add_dataframe_table(
        document,
        dataframe,
        font_size=5.5,
    )

    included_files.append(filename)


# =====================================================================
# 10. SAVE
# =====================================================================

document.save(OUTPUT_WORD_FILE)

print("\nWORD REPORT CREATED SUCCESSFULLY")
print("=" * 75)
print("Saved to:")
print(OUTPUT_WORD_FILE)

print("\nIncluded result files:")

for filename in included_files:
    print("  INCLUDED:", filename)

if missing_files:
    print("\nFiles not found and skipped:")

    for filename in missing_files:
        print("  MISSING:", filename)

ModuleNotFoundError: No module named 'docx'